## Towards Fairness in Multi-modal Chest X-Ray Diagnosis

## Project Overview
This project develops an unbiased AI diagnostic system for chest X-ray analysis that maintains consistent performance across diverse patient demographics (age, gender) and imaging protocols (AP/PA views). The focus is on three thoracic pathologies: **Atelectasis**, **Cardiomegaly**, and **Effusion**.


#### Main Tasks
#### a) Multi-Pathology Classification

**Objective:** Build and evaluate a model that classifies X-ray images for three specific thoracic pathologies while maintaining consistent performance across all patient groups.

#### b) Bias Mitigation & Robustness Enhancement

**Objective:** Implement techniques to ensure equal performance while handling complex cases effectively.


### What we are detecting:


| Pathology | What it is | Challenges |
|-----------|------------|------------|
| **Atelectasis** | Collapsed lung area | **Elderly**: Often after surgery<br>**Young**: Usually from blockages |
| **Cardiomegaly** | Enlarged heart | Heart size standards change with age and gender |
| **Effusion** | Fluid in lungs | Looks different in AP vs PA X-ray views |

## Dataset Source: NIH ChestX-ray-14 Dataset

We are utilizing a curated subset from the **NIH ChestX-ray-14 dataset**, a large-scale medical imaging collection widely recognized in healthcare AI research.

**Primary Source:** [NIH Clinical Center](https://nihcc.app.box.com/v/ChestXray-NIHCC)  
**Accessible Subset:** [NIH Chest X-rays Sample on Kaggle](https://www.kaggle.com/datasets/nih-chest-xrays/sample?select=sample)

### Dataset Overview:
The NIH ChestX-ray-14 dataset is a comprehensive collection of chest X-ray images featuring:
- **Large-scale Collection**: Extensive dataset with over 100,000 frontal-view X-ray images
- **Rich Annotations**: Each image includes detailed patient information and disease labels
- **Multiple Pathologies**: Comprehensive labeling across 14 common thoracic conditions
- **Research Standard**: Widely adopted benchmark in medical AI and computer vision research


### Dataset Characteristics:
- **Demographic Diversity**: Includes patient age, gender, and imaging view metadata
- **Clinical Relevance**: Real-world medical images with expert annotations
- **Research Validation**: Established benchmark for developing and evaluating diagnostic AI models
- **Quality Assurance**: Curated sample maintaining data integrity and clinical relevance
- **Pre-defined Splits**: Dataset is already partitioned into training, validation, and test sets

### Dataset Structure:
The subset of the dataset comes with pre-built splits to ensure proper model evaluation:
- **Training Set**: Used for model development and parameter learning
- **Validation Set**: Used for hyperparameter tuning and model selection
- **Test Set**: Reserved for final evaluation on unseen data

| Column | What It Contains | Importance for Your Project |
|--------|------------------|----------------------------|
| **Image Index** | Filename of the X-ray image | Links images to metadata for data processing |
| **Finding Labels** | Pathology diagnoses (multiple possible) | Target variables for multi-pathology classification |
| **Patient ID** | Unique patient identifier | Crucial for creating patient-wise splits to prevent data leakage |
| **Patient Age** | Age in format like "060Y" (60 years) | Key demographic factor for fairness analysis across age groups |
| **Patient Gender** | M/F | Demographic factor for ensuring gender equity in model performance |
| **View Position** | AP/PA (X-ray view type) | Technical factor for model robustness across different imaging protocols |

## 1. Setup and Installation <a id="setup"></a>
### Install Required Libraries

We install and import required libraries, run this once per new environment.

In [ ]:
#@title import libraries (2 minutes)
# Library Installations
!pip install pydicom SimpleITK albumentations torchmetrics grad-cam -q

# Core Library Imports
import os
import sys, math, random, time, warnings
from glob import glob
from pathlib import Path
import gc

# Data Handling and Processing
import numpy as np
import pandas as pd
import cv2
import pydicom
from PIL import Image

# Deep Learning with PyTorch & Torchvision
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms, models
from torch.optim.lr_scheduler import OneCycleLR

# Visualization & Metrics
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score, RocCurveDisplay
from torchmetrics.classification import BinaryAUROC

# Explainability (XAI)
from pytorch_grad_cam import GradCAM, ScoreCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image

# Environment Configuration
warnings.filterwarnings("ignore")

# Ensure reproducibility
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

RANDOM_SEED = 42
seed_everything(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Setup complete. Using device: {device}")

## 2. Baseline Understanding

#### As mentioned before, we will be using a subset of this data:

| Metric | Value | Significance |
| :--- | :--- | :--- |
| Total Images | 5,606 chest X-rays | Substantial dataset for meaningful analysis |
| Total Patients | 4,230 unique individuals | Good patient diversity |
| Images per Patient | 1.33 average | Minimal patient repetition |
| Dataset Size | ~2.25 GB | Manageable for student projects |
| Pre-split | Yes | No data leakage concerns |

#### Split Distribution:
| Split | Images | Percentage | Patients |
| :--- | :--- | :--- | :--- |
| Training | 3,884 | 69.3% | 2,961 |
| Validation | 578 | 10.3% | 423 |
| Test | 1,144 | 20.4% | 846 |

#### Learning Implications

- **Will need techniques for handling severe class imbalance**
- **Fairness analysis possible across all demographic dimensions** 
- **Sufficient data volume for meaningful model training and evaluation**

## Essential First Steps

- Load the test/train/val files provided to you.
- Verify dataset loading and confirm basic statistics match expected values
- Confirm no patient overlap between train/val/test splits  
- Visualize pathology distributions across demographic groups
- Identify potential bias sources in the data distribution

## Key Questions to Answer

- Where are the biggest class imbalances?
- Which demographic groups have the least representation?
- How do pathology rates vary by age, gender, and view position?

In [ ]:
# FOCUS 1: Dataset loading and basic exploration
def load_and_explore_dataset():
    """Students should understand dataset structure and splits"""
    
    # Load pre-split datasets
    train_df = pd.read_csv('train_df.csv')
    val_df = pd.read_csv('val_df.csv') 
    test_df = pd.read_csv('test_df.csv')
    
    # KEY INSIGHT: Always check dataset sizes first
    print("Dataset Sizes:")
    print(f"Training set: {len(train_df)} images")
    print(f"Validation set: {len(val_df)} images") 
    print(f"Test set: {len(test_df)} images")
    
    # Combine for exploration (but maintain splits for training)
    all_data = pd.concat([train_df, val_df, test_df], ignore_index=True)
    
    return train_df, val_df, test_df, all_data

# EXECUTE
train_df, val_df, test_df, all_data = load_and_explore_dataset()

# Focus 2: Dataset Loading and basic exploration

# Focus 3: Demographic analysis for fairness assessment

# Focus 4: Class imbalance analysis across demographics

## 2. Data-Preprocessing 

### Key Learning Objectives

- **Prepare data for foundation models**
- **Handle medical imaging specific preprocessing** 
- **Create reproducible preprocessing pipelines**

### Essential Steps
1. **Data Validation**
   - Verify image quality and consistency
   - Confirm label accuracy and completeness
   - Check for corrupted files

2. **Medical Imaging Preprocessing**
   - Standardize image dimensions and orientation
   - Normalize pixel intensity values

3. **Pipeline Development**
   - Create modular preprocessing functions
   - Ensure reproducibility across splits
   - Document all transformation steps

# 3. Feature-Based Classification with Pretrained Foundation Models

## Step 1: Feature Extraction

**Objective**: Use pre-trained foundation models as fixed feature extractors

**Foundation Models** are large-scale AI models pre-trained on massive datasets that can be adapted to various downstream tasks. Think of them as "general-purpose" AI that already understands basic patterns and can be specialized for your specific needs.

**Process**:
- Load pre-trained medical foundation models (DenseNet, EfficientNet, BiomedCLIP, etc.)
- Pass X-ray images through the model **without training/updating weights**
- Extract the final feature vectors before classification layers
- Save these feature representations for training

**Key Concept**: The foundation model acts as a sophisticated feature engineering tool that understands medical image patterns without any fine-tuning.

## Step 2: Classification Head Training

**Objective**: Train only the final layers for specific pathology diagnosis

**Process**:
- Take extracted features from Step 1 as input
- Build a simple classifier on top (typically 1-3 dense layers)
- Train **only these new layers** while keeping foundation model frozen
- Use binary/multi-label classification for targeted pathologies

**Key Concept**: This is efficient and prevents overfitting since we're training very few parameters on limited medical data.


## Step 3: Baseline Performance Establishment

**Objective**: Measure how well the foundation model transfers to your specific task

**Process**:
- Evaluate on held-out test set
- Calculate standard metrics (accuracy, precision, recall, F1-score)
- Compare against simple benchmarks (random, majority class)
- This becomes your performance baseline for future experiments

**Key Concept**: This baseline tells you what's achievable with minimal training effort using pre-trained knowledge.


## Step 4: Subgroup Performance Analysis

**Objective**: Understand how performance varies across patient demographics

**Process**:
- Break down results by age groups, gender, ethnicity, hospital sites
- Calculate performance metrics for each subgroup separately
- Identify any significant performance disparities
- Document where the model works well and where it struggles

**Key Concept**: Medical AI must work equitably across all patient populations - subgroup analysis reveals potential biases.

## Step 5: Output and Documentation

**Objective**: Create comprehensive baseline report

**Deliverables**:
- Baseline accuracy and performance metrics
- Subgroup performance breakdown tables
- Confusion matrices for each pathology
- Feature importance analysis
- Recommendations for model improvement

**Key Concept**: This baseline becomes the reference point for all future model development and improvement efforts.


### Some Foundational Models you can use (or research more on your own)

#### 1. DenseNet-121

In [ ]:
import torchvision.models as models
import torch.nn as nn

model = models.densenet121(pretrained=True)
# Freeze backbone for feature extraction
for param in model.parameters():
    param.requires_grad = False
# Replace classifier for your task
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, 3)  # 3 pathologies

#### 2. BiomedCLIP 

In [ ]:
# 2. BiomedCLIP - Example (requires transformers library)
from transformers import AutoModel, AutoProcessor

model = AutoModel.from_pretrained("microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
processor = AutoProcessor.from_pretrained("microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")

#### 3. Rad-DINO

In [ ]:
# 3. EfficientNet
model = models.efficientnet_b0(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
# Replace classifier
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 3)

#### 4. Medical Vision Transformers

In [ ]:
# MedImageInsight / Medical ViT variants
model = tf.keras.applications.ViT_B16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

#### 5. EfficientNet Medical

In [ ]:
# MedImageInsight / Medical ViT variants
model = tf.keras.applications.ViT_B16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

#### 6. MONAI Model Zoo Models

In [ ]:
# MONAI provides medical pre-trained models
from monai.networks.nets import DenseNet121, EfficientNetBN

# DenseNet for medical imaging
model = DenseNet121(
    spatial_dims=2,
    in_channels=3,
    out_channels=num_classes
)

# Load pre-trained weights
# weights = torch.hub.load_state_dict_from_url(MODEL_URL)

## Part 4: Parameter-Efficient Fine-Tuning & Performance Comparison

### What You'll Learn in Part 4

In this section, you'll transform your pre-trained foundation model into a specialized medical AI assistant. We'll use **parameter-efficient fine-tuning** to adapt the model to chest X-rays while preserving its general knowledge.


### Understanding Parameter-Efficient Fine-Tuning

### **Why Not Train Everything?**

| Approach | Parameters Updated | Training Time | Risk of Overfitting |
|----------|-------------------|---------------|---------------------|
| **Train From Scratch** | 100% | Days | Very High |
| **Full Fine-Tuning** | 100% | Hours | High |
| **Parameter-Efficient** | 1-5% | Minutes | Low |

### **The Smart Strategy:**

```python
# Instead of this (inefficient):
for param in model.parameters():
    param.requires_grad = True  # Update everything

# We do this (smart):
for param in model.parameters():
    param.requires_grad = False  # Freeze most layers
    
for param in model.classifier.parameters():
    param.requires_grad = True   # Only update classifier

## Step 1: Setup Parameter-Efficient Fine-Tuning

After establishing your baseline, this step focuses on strategic model adaptation that maximizes performance while minimizing computational cost and potential bias amplification. Efficient fine-tuning allows you to specialize foundation models for chest X-ray analysis without "forgetting" their general knowledge.

Instead of training all millions of parameters in a foundation model, we strategically choose which parts to update, balancing:

- Performance: Adapting to medical domain specifics
- Efficiency: Reducing training time and resources
- Stability: Preserving general feature knowledge
- Fairness: Preventing overfitting to majority groups

In [ ]:
def setup_efficient_fine_tuning(model, strategy="classifier_only"):
    """
    Configure a pre-trained model for parameter-efficient fine-tuning.
    
    This function implements three strategies for efficient fine-tuning by
    selectively unfreezing specific parts of the model while keeping the
    majority of parameters frozen.
    
    Parameters:
    -----------
    model : torch.nn.Module
        Pre-trained model to be configured for fine-tuning
    strategy : str
        Fine-tuning strategy to use. Options:
        - 'classifier_only': Only train the final classification layers
        - 'last_block': Train the last feature block and classifier
        - 'lora': Use Low-Rank Adaptation (requires additional implementation)
    
    Returns:
    --------
    torch.nn.Module
        Model configured for the specified fine-tuning strategy
    """
    
    # Freeze all model parameters initially
    for param in model.parameters():
        param.requires_grad = False
    
    if strategy == "classifier_only":
        # Only unfreeze the classifier/fully-connected layers
        for param in model.classifier.parameters():
            param.requires_grad = True
        print("Strategy: Training only classifier layers")
        
    elif strategy == "last_block":
        # Unfreeze the last feature extraction block and classifier
        for param in model.features.denseblock4.parameters():
            param.requires_grad = True
        for param in model.classifier.parameters():
            param.requires_grad = True
        print("Strategy: Training last feature block and classifier")
        
    elif strategy == "lora":
        # Apply Low-Rank Adaptation
        # Note: This requires additional LoRA implementation
        model = apply_lora_adapters(model)
        print("Strategy: Using Low-Rank Adaptation (LoRA)")
    
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Choose from 'classifier_only', 'last_block', or 'lora'")
    
    return model


def apply_lora_adapters(model):
    """
    Apply Low-Rank Adaptation to the model.
    
    Note: This is a placeholder function. Actual LoRA implementation
    would require more complex modifications to the model architecture.
    """
    # This would typically involve:
    # 1. Identifying target layers for LoRA adaptation
    # 2. Adding low-rank matrices parallel to existing layers
    # 3. Setting up the forward pass to include LoRA contributions
    
    print("Note: LoRA implementation required for full functionality")
    return model


# Example usage with different model architectures
def setup_resnet_fine_tuning(model, strategy="classifier_only"):
    """
    Example for ResNet architecture - adjusts for different layer names
    """
    for param in model.parameters():
        param.requires_grad = False
    
    if strategy == "classifier_only":
        for param in model.fc.parameters():
            param.requires_grad = True
        print("ResNet - Training only fully-connected layer")
        
    elif strategy == "last_block":
        # Unfreeze layer4 (last ResNet block) and fc layer
        for param in model.layer4.parameters():
            param.requires_grad = True
        for param in model.fc.parameters():
            param.requires_grad = True
        print("ResNet - Training last residual block and fully-connected layer")
    
    return model


# Utility function to count trainable parameters
def count_trainable_parameters(model):
    """
    Count the number of trainable parameters in the model
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Example usage and testing
if __name__ == "__main__":
    import torchvision.models as models
    
    # Load a pre-trained model
    model = models.densenet121(pretrained=True)
    total_params = sum(p.numel() for p in model.parameters())
    
    print(f"Total parameters in model: {total_params:,}")
    
    # Test different strategies
    strategies = ['classifier_only', 'last_block']
    
    for strategy in strategies:
        test_model = models.densenet121(pretrained=True)
        configured_model = setup_efficient_fine_tuning(test_model, strategy)
        trainable_params = count_trainable_parameters(configured_model)
        
        print(f"{strategy}: {trainable_params:,} trainable parameters "
              f"({trainable_params/total_params*100:.2f}% of total)")

## Step 2: Implement Fairness-Aware Training
Main goal of this step is to train the AI to work equally well for all patient groups - not just average performance.

In [ ]:
class FairnessAwareTrainer:
    def __init__(self, model, demographics=['age', 'gender']):
        self.model = model
        self.demographics = demographics
        
    def train_epoch(self, dataloader, optimizer, criterion):
        
        self.model.train()
        total_loss = 0
        
        for batch_idx, (images, labels, demo_info) in enumerate(dataloader):
            
            # Forward pass
            outputs = self.model(images)
            
            # Calculate loss with fairness penalty
            loss = criterion(outputs, labels, demo_info)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # Monitor fairness every 20 batches
            if batch_idx % 20 == 0:
                self.log_fairness_metrics(outputs, labels, demo_info)
        
        return total_loss / len(dataloader)

## Step 3: Model Evaluation and Bias Assessment

After implementing fairness-aware training techniques, you'll conduct a thorough evaluation to understand how your model performs across different patient groups and whether your interventions were effective.

- Evaluating your fine-tuned model on the test set
- Analyzing performance across all demographic subgroups
- Comparing results with your baseline to measure improvement
- Identifying any remaining fairness issues

In [ ]:
def comprehensive_fairness_evaluation(model, test_dataset, test_df, demographics):
    """
    Evaluate model performance across all demographic subgroups
    """
    
    print("Evaluating model fairness across demographcis...")
    
    # Get predictions
    test_predictions = model.predict(test_dataset)
    
    # Initialize results storage
    subgroup_results = {}
    fairness_metrics = {}
    
    # Evaluate for each demographic factor
    for demographic in demographics:
        print(f"\n Analyzing {demographic.upper()} ")
        subgroup_results[demographic] = {}
        
        # Get unique values for this demographic (e.g., ['M', 'F'] for gender)
        groups = test_df[demographic].unique()
        
        for group in groups:
            # Create mask for this subgroup
            mask = test_df[demographic] == group
            subgroup_size = mask.sum()
            
            if subgroup_size > 0:  # Ensure we have samples
                subgroup_metrics = calculate_subgroup_metrics(
                    test_df[mask], test_predictions[mask]
                )
                subgroup_results[demographic][group] = {
                    'metrics': subgroup_metrics,
                    'sample_size': subgroup_size
                }
                
                print(f"  {group} (n={subgroup_size}):")
                print(f"    AUC: {subgroup_metrics['avg_auc']:.3f}")
                print(f"    F1: {subgroup_metrics['avg_f1']:.3f}")
    
    return subgroup_results

def calculate_subgroup_metrics(subgroup_df, subgroup_predictions):
    """
    Calculate comprehensive metrics for a specific subgroup
    """
    from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
    
    pathologies = ['Atelectasis', 'Cardiomegaly', 'Effusion']
    metrics = {}
    
    for i, pathology in enumerate(pathologies):
        true_labels = subgroup_df[pathology].values
        pred_scores = subgroup_predictions[:, i]
        pred_binary = (pred_scores > 0.5).astype(int)
        
        # Handle cases where a pathology might not appear in subgroup
        if len(np.unique(true_labels)) > 1:
            auc = roc_auc_score(true_labels, pred_scores)
            f1 = f1_score(true_labels, pred_binary, zero_division=0)
            precision = precision_score(true_labels, pred_binary, zero_division=0)
            recall = recall_score(true_labels, pred_binary, zero_division=0)
        else:
            auc = f1 = precision = recall = 0.0
            
        metrics[pathology] = {
            'auc': auc, 'f1': f1, 'precision': precision, 'recall': recall
        }
    
    # Calculate averages
    metrics['avg_auc'] = np.mean([metrics[p]['auc'] for p in pathologies])
    metrics['avg_f1'] = np.mean([metrics[p]['f1'] for p in pathologies])
    
    return metrics

**Task:**
- Create a dashboard that shows real-time performance across subgroups
- Set up alerts when performance disparities exceed your threshold

**Task:**
- Experiment with fairness penalty weights (0.1, 0.3, 0.5)
- Try different fairness definitions (demographic parity, equalized odds)


## Step 6: Comprehensive Evaluation

### What Makes a Good Medical AI Model?

- **High overall accuracy**  
  The model should perform well across common clinical cases.

- **Good performance on rare conditions**  
  It must correctly identify and handle low-prevalence diseases, not just the common ones.

- **Equitable across all patient groups**  
  Performance should be consistent across demographics such as age, sex, ethnicity, and socioeconomic status.

- **Reliable and consistent**  
  Outputs should be stable, reproducible, and trustworthy in real-world settings.

## Evaluation Script

In [ ]:
def comprehensive_evaluation(model, test_loader):
    results = {
        'overall': calculate_metrics(model, test_loader),
        'subgroups': {}
    }
    
    # Analyze each demographic subgroup
    for subgroup in ['young', 'middle', 'elderly', 'male', 'female']:
        subgroup_loader = get_subgroup_loader(test_loader, subgroup)
        results['subgroups'][subgroup] = calculate_metrics(model, subgroup_loader)
    
    # Calculate fairness metrics
    results['fairness'] = calculate_fairness_metrics(results['subgroups'])
    
    return results


## Step 7: Complete Pipeline Implementation


In [ ]:
def run_fine_tuning_pipeline():
    # 1. Setup
    model = MedicalModel(num_classes=8)
    train_loader = create_fair_dataloader(train_data)
    
    # 2. Training loop
    for epoch in range(50):
        setup_training_phases(model, epoch)
        
        # Training with fairness monitoring
        train_epoch(model, train_loader, FairnessLoss())
        
        # Validation and logging
        if epoch % 5 == 0:
            results = comprehensive_evaluation(model, val_loader)
            log_results(epoch, results)
    
    return model


**Task**
- Run the full pipeline end-to-end
- Document fairness improvements over time
- Present findings: accuracy–fairness trade-offs

# 4. Comprehensive Model Evaluation & Fairness Assessment

Your task is to conduct a comprehensive evaluation of your chest X-ray classification model to answer these critical questions:

- How well does your model actually work? - Overall performance
- Does it work equally well for ALL patients? - Fairness across demographics
- Where does it fail and why? - Error analysis and interpretation

Required Analysis Outputs:

- Performance Metrics Table - AUC, precision, recall, F1 for all pathologies
- Subgroup Analysis Report - Performance broken down by age, gender, view position
- Fairness Gap Calculation - Maximum performance differences between groups
- Statistical Significance Results - p-values for observed differences

The final part in this project conducts a complete evaluation of your framework.

| Evaluation Level       | Metrics                             | Purpose                                             |
|------------------------|--------------------------------------|-----------------------------------------------------|
| Overall Performance    | AUC-ROC, Precision, Recall, F1       | Model effectiveness across entire dataset           |
| Subgroup Analysis      | Stratified metrics by demographics   | Identify performance variations across patient groups |
| Fairness Assessment    | Performance gaps, statistical significance | Quantify and validate equity across subgroups   |
| Model Interpretation   | Grad-CAM, attention visualization    | Understand model decision-making process            |


In [ ]:
def evaluation_pipeline(model, test_dataset, test_df, pathologies, demographics):
    """
    Complete evaluation across all metrics and subgroups
    """
    print("Model Fairness Evaluation Pipelines")
    
    # Get model predictions
    test_predictions = model.predict(test_dataset)
    
    # 1. Overall Performance Metrics
    overall_metrics = calculate_overall_metrics(test_df, test_predictions, pathologies)
    
    # 2. Stratified Subgroup Analysis
    subgroup_metrics = analyze_subgroup_performance(test_df, test_predictions, pathologies, demographics)
    
    # 3. Fairness Gap Analysis
    fairness_report = calculate_fairness_gaps(subgroup_metrics)
    
    # 4. Statistical Significance Testing
    significance_results = statistical_significance_analysis(test_df, test_predictions, demographics, pathologies)
    
    return {
        'overall_metrics': overall_metrics,
        'subgroup_metrics': subgroup_metrics,
        'fairness_report': fairness_report,
        'significance_results': significance_results
    }


## Key Metrics for Each Pathology

| Metric     | Formula                                      | Clinical Importance                         |
|------------|-----------------------------------------------|----------------------------------------------|
| AUC-ROC    | Area under ROC curve                          | Overall diagnostic accuracy                  |
| Precision  | TP / (TP + FP)                                | How reliable positive predictions are        |
| Recall     | TP / (TP + FN)                                | Ability to find all positive cases           |
| F1-Score   | 2 × (Precision × Recall) / (Precision + Recall) | Balance between precision and recall       |



# Part 5: Comprehensive Visualization Suite

## Visualization Objectives
Create a complete visual analysis package that reveals model performance, fairness, and decision-making processes across all patient subgroups.

## Core Visualization Components

### 1. Demographic Analysis Plots
**Purpose**: Understand your patient population distribution

## What you'll create:
- Age distribution histograms
- Gender balance pie charts
- Race/ethnicity breakdowns
- Clinical view position distributions
- Comorbidity prevalence across groups

### 2. Performance Comparison Across Subgroups
**Purpose**: Measure fairness and equity in model performance

## Multi-level analysis:
- Accuracy bar charts by age/gender/race
- F1-score comparisons across subgroups
- Precision-Recall tradeoffs by demographic
- ROC curves for each pathology + subgroup combination

### 3. Confusion Matrices by Subgroup
**Purpose**: Measure fairness and equity in model performance

## Matrix comparisons:
- Side-by-side confusion matrices for each demographic
- Error type analysis (false positives vs false negatives)
- Pathology-specific mistake patterns

### 3. Grad-CAM Heatmaps
**Purpose**: See where the model "looks" in medical images

## Model attention visualization:
- Heatmaps overlaid on original X-rays
- Comparison of attention between correct vs incorrect predictions
- Demographic patterns in model focus areas

In [ ]:
def plot_demographic_distribution(df):
    """Visualize patient demographics"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Age distribution
    ages = df['Patient Age'].str.extract('(\d+)').astype(int)
    axes[0, 0].hist(ages, bins=20, edgecolor='black')
    axes[0, 0].set_title('Age Distribution')
    
    # Gender distribution
    df['Patient Gender'].value_counts().plot(kind='pie', ax=axes[0, 1], autopct='%1.1f%%')
    axes[0, 1].set_title('Gender Distribution')
    
    # View position
    df['View Position'].value_counts().plot(kind='bar', ax=axes[1, 0])
    axes[1, 0].set_title('View Position Distribution')
    
    # Pathology prevalence
    pathologies = ['Atelectasis', 'Cardiomegaly', 'Effusion']
    prevalence = [df[p].sum() for p in pathologies]
    axes[1, 1].bar(pathologies, prevalence)
    axes[1, 1].set_title('Pathology Prevalence')
    
    plt.tight_layout()
    plt.show()

def plot_subgroup_performance(subgroup_results, metric='auc'):
    """Compare performance across demographic subgroups"""
    # Implementation needed
    pass

def visualize_gradcam(model, image, target_layer, class_idx):
    """Show Grad-CAM heatmap for model decision"""
    # Implementation needed
    pass

## Research Questions

#### 1. Does the model perform equally well for male and female patients?
#### 2. How much does fine-tuning improve performance over using pre-trained features as-is?
#### 3. Where does the model "look" in X-ray images when making decisions?
#### 4. Are there certain diseases the model detects better for some groups than others?
#### 5. How much does fairness-aware training reduce performance gaps between groups?

| **Deliverable** | **Description** |
|-----------------|-----------------|
| **1. Google Colab Notebook (.ipynb)** | - Clean, well-commented end-to-end workflow: data exploration → preprocessing → model development → evaluation<br>- Visualisations: ROC curves, confusion matrices, Grad-CAM heatmaps, demographic performance comparisons |
| **2. Technical Report (3–5 pages)** | - Methodology: preprocessing decisions, model architecture, training approach<br>- Results: performance metrics (AUC-ROC, precision, recall, F1) stratified by demographics<br>- Fairness assessment: analysis of subgroup performance gaps<br>- Discussion: technical challenges, limitations, ethical considerations in medical AI |
| **3. Oral Presentation (10 minutes)** | - Technical approach and implementation decisions<br>- Results with emphasis on fairness across demographics<br>- Optimisation and bias-mitigation strategies<br>- Reflections on responsible AI in healthcare |
